In [1]:
import re
from docling.document_converter import DocumentConverter,PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend
from docling.datamodel.base_models import InputFormat
import os
pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr = False
pipeline_options.do_table_structure = True
pipeline_options.table_structure_options.do_cell_matching = False

In [2]:
def pdf2md_single(source) :
    converter = DocumentConverter(format_options={ InputFormat.PDF: PdfFormatOption(
                pipeline_options=pipeline_options, backend=PyPdfiumDocumentBackend
            )
        }
    )
    result = converter.convert(source)
    return result.document.export_to_markdown()


#file = open('pdf2md.md','w',encoding='utf-8')
#file.write(pdf2md_single("../unique_sxolika_pdf/paper_16.pdf"))

In [35]:
def paragraph_maker(text,maxpadding = 2) :
    lines = text.splitlines()
    paragraphs = []
    emptyline = 0
    paragraph = ''
    for i,line in enumerate(lines) :
        if i == len(lines) - 1 :
            paragraph = paragraph + line
            paragraphs.append(paragraph)
            continue
        if line == '' :
            emptyline = emptyline + 1
            if emptyline == maxpadding :
                if paragraph != '' :
                    paragraphs.append(paragraph)
                emptyline = 0
                paragraph = ''
        else :
            paragraph = paragraph + line + '\n'
            emptyline = 0
            if i == len(lines) - 1 :
                paragraphs.append(paragraph)
    return paragraphs

def paragraph_merger(paragraphs,min_par_size,threshold) :
    newparagraphs = []
    for i,paragraph in enumerate(paragraphs) :
        if len(paragraph) < min_par_size :
            if paragraph != paragraphs[-1] :
                if len(paragraphs[i+1]) > threshold :
                    if not (paragraphs[i+1].startswith('##') or paragraphs[i+1].startswith('|') ) :
                        paragraphs[i+1] = paragraph + paragraphs[i+1]
                        continue
        newparagraphs.append(paragraph)
    return newparagraphs

def paragraph_clean_image(paragraphs) :
    newparagraphs = []
    for paragraph in paragraphs :
        if paragraph.startswith('<!-- image -->') :
            continue
        else :
            newparagraphs.append(paragraph)
    return newparagraphs

def paragraph_clean_dotlines(paragraphs) :
    newparagraphs = []
    for paragraph in paragraphs :
        if paragraph.startswith('.....') :
            if paragraph.endswith('.....') :
                continue
        else :
            newparagraphs.append(paragraph)
    return newparagraphs

def paragraph_remove_artifacts(paragraphs,min_length = 5) :
    newparagraphs = []
    for paragraph in paragraphs :
        if len(paragraph) <= min_length :
            continue
        newparagraphs.append(paragraph)
    return newparagraphs

def paragraph_not_char_end(paragraph,chars,print) :
    #if not paragraph.startswith('##') :
    end_search_flag = False
    if not paragraph.endswith(chars) :
        if print :
            paragraph = '[Out:Paragraph does not end with specified char]' + paragraph
        else : paragraph = ''
        end_search_flag = True
    return (paragraph,end_search_flag)

def all_paragraph_not_char_end(paragraphs,chars,print = False) :
    newparagraphs = []
    for i,paragraph in enumerate(paragraphs) :
        newparagraphs.append(paragraph_not_char_end(paragraph,chars,print)[0])
        if paragraph_not_char_end(paragraph,chars,print)[1] == False :
            for t_paragraph in paragraphs[i+1:] :
                newparagraphs.append(t_paragraph)
            return newparagraphs
    return newparagraphs

def remove_content_table_begin(paragraphs,num_of_front = 10,print = False) :
    newparagraphs = []
    for paragraph in paragraphs[:num_of_front] :
        if paragraph.startswith('|') and paragraph.endswith('|\n') :
            if print :
                paragraph = '[Out:Conent Table]' + paragraph
                newparagraphs.append(paragraph)
        else :
            newparagraphs.append(paragraph)
    for paragraph in paragraphs[10:] :
        newparagraphs.append(paragraph)
    return newparagraphs

def paragraph_fix_broken_line(paragraphs) :
    ending_lower_pattern = re.compile(r'(.*[α-ωά-ώa-zΑ-ΩΆ-Ώ]\n)|(.*[0-9]\n)|(.*°C\n)|(.*\)\n)|(.*\-\n)|(.*,\n)')
    starting_lower_pattern = re.compile(r'([α-ωά-ώa-z])|([0-9])|(°)|\)|- ')
    newparagraphs = []
    for i,paragraph in enumerate(paragraphs) :
        if re.match(ending_lower_pattern,paragraph) :
            if paragraph != paragraphs[-1] :
                if re.match(starting_lower_pattern,paragraphs[i+1]) :
                    paragraphs[i+1] = paragraph + paragraphs[i+1]
                    continue
        newparagraphs.append(paragraph)
    return newparagraphs

def print_text(paragraphs) :
    for paragraph in paragraphs :
        print(paragraph)
        print('\n\n\n')

def remove_ending_chunk(paragraphs) :
    paragraphs = paragraphs[::-1]
    for i,paragraph in enumerate(paragraphs) :
        if paragraph.startswith('##') :
            paragraphs[i] = '[Out:Paragraph is in ending chunk]' + paragraphs[i]
            return paragraphs[::-1]
        paragraphs[i] = '[Out:Paragraph is in ending chunk]' + paragraphs[i]
        #del paragraphs[-1]
    return paragraphs[::-1]

def remove_funding(paragraphs) :
    funding_pattern = re.compile(r'συγχρηματοδοτείται από την Ελλάδα και την Ευρωπαϊκή Ένωση|συγχρηµατοδοτείται από την Ελλάδα και την Ευρωπαϊκή Ένωση')
    newparagraphs = []
    for paragraph in paragraphs :
        if re.search(funding_pattern,paragraph) :
            continue
        newparagraphs.append(paragraph)
    return newparagraphs

def remove_taged_paragraphs(paragraphs,tags,print = False) :
    newparagraphs = []
    bibliography_flag = False
    for paragraph in paragraphs :
        if paragraph.startswith(tags) :
            bibliography_flag = True
            if print :
                paragraph = '[Out:Paragraph is' + tags[0] + ']' + paragraph
            continue
        elif bibliography_flag == True :
            if paragraph.startswith('##') and not paragraph.startswith(tags) :
                bibliography_flag = False
                newparagraphs.append(paragraph)
                continue
            if print :
                paragraph = '[Out:Paragraph is' + tags[0] + ']' + paragraph
            else : continue
        newparagraphs.append(paragraph)
    return newparagraphs

def remove_legal_note_3966_2011(paragraphs,print = False) :
    newparagraphs = []
    for paragraph in paragraphs :
        if re.search('ν. 3966/2011',paragraph):
            if print :
                paragraph = '[Out:Legal note 3966/2011]' + paragraph
            else : continue
        newparagraphs.append(paragraph)
    return newparagraphs

def remove_diofantos(paragraphs,print = False) :
    newparagraphs = []
    for paragraph in paragraphs :
        if re.search('IΤΥΕ - ΔΙΟΦΑΝΤΟΣ',paragraph) :
            if print :
                paragraph = '[Out:Diofantos Publisehr Note]' + paragraph
            else : continue
        newparagraphs.append(paragraph)
    return newparagraphs

def write_text(paragraphs,file) :
    text = '[Paragraph Begin: Number 0 ]'
    for i,paragraph in enumerate(paragraphs) :
        text = text + paragraph + '[Paragraph End]\n\n\n[Paragraph Begin: Number ' + str(i+1) + ' ]'
    file.write(text) 

In [36]:
os.makedirs('../test_output', exist_ok=True)
for i,file in enumerate(os.listdir('../md_output')) :
    if not file.endswith('.md'):
            continue
    try:
        with open('../md_output'+'/'+file, 'r', encoding='utf-8') as infile:
            text = infile.read()
    except Exception as e:
        print(f"Error reading {file}: {e}")
        continue

    endings = ('.\n','?\n','!\n',';\n','.\n',';\n','|\n')
    bibliography_tags = ('## ΞΕΝΟΓΛΩΣΣΗ','## ΕΛΛΗΝΙΚΗ','## Bl ΒΛΙΟΓΡΑΦΙΑ', '## ΒΛΙΟΓΡΑΦΙΑ','## Π Ι ΝΑ Κ Α Σ Π Ρ Ο Ε Λ Ε Υ Σ Η Σ Τ Ω Ν E Ι ΚΟ Ν Ω Ν',
                         '## ΠΙΝΑΚΑΣ ΠΡΟΕΛΕΥΣΗΣ ΤΩΝ EΙΚΟΝΩΝ','## ΒΙΒΛΙΟΓΡΑΦΙΑ','## ΕΙΚΟΝΟΓΡΑΦΙΚΟ ΥΛΙΚΟ','## Διαδικτυακοί τόποι','## ΚΕΙΜΕΝΑ',
                         '## Ηλεκτρονικές διευθύνσεις στο Internet','## Ξενόγλωσση','## Ελληνική','## ΠΑΡΑΡΤΗΜΑ','## Στ. Φωτογραφικά άλμπουμ',
                         '## Θ. Ιστότοποι','## www.','## Η. Λοιπές μελέτες','## Επιλογή Βιβλιογραφίας','## ΠΑΙΔΑΓΩΓΙΚΕΣ ΜΕΛΕΤΕΣ','## ΠΗΓΕΣ ΦΩΤΟΓ ΡΑΦΙΚΟΥ ΥΛΙΚΟΥ',
                         '## Βιβλιογραφία','## 7. Εικονική βιβλιοθήκη Επιφανειοδραστικών','## ΞΕΝΗ ΒΙΒΛΙΟΓΡΑΦΙΑ','## Βιβλιογραφικά Βοηθήματα',)
    glossary_tags = ('## ΓΛΩΣΣΑΡΙ, ## Γλωσσάρι','## Λ EΞIKO','## ΛEΞIKO','## ΛΕΞΙΛΟΓΙΟ','## ΓΛΩΣΣΑ ΡΙ','## Ζ. Λε ξ ικά','## Γλωσσάρι')
    euritirio_tags = ('## EYPETHPIA','## ΕΥΡΕΤΗΡΙΟ ΟΡΩΝ','## Ευρετήριο όρων')
    church_tags = ('## Δ. Λειτουργικά Β ι β λία','## Β . Μουσικά β ι β λία','## Ε. Πατερικά Έργα','## e-kere.gr','## Ι. Π η γές Εικόνων',
                   '## Α\' Γ Υ Μ Ν Α Σ Ι Ο Υ')
    content_tags = ('## ΠΕΡΙΕΧΟΜΕΝΑ','## ΠΕΡΙΕΧΟΜΕΝΟ','## Περιεχόμενο','## Περιεχόμενα','## ΠΙΝΑΚΑΣ ΠΕΡΙΕΧΟΜΕΝΩΝ')
    printing_tags = ('## ΣΤΟΙΧΕΙΑ ΕΠΑΝΕΚ∆ΟΣΗΣ','## ΣΤΟΙΧΕΙΑ ΑΡΧΙΚΗΣ ΕΚ∆ΟΣΗΣ','## ΥΠΟΥΡΓΕΙΟ ΠΑΙΔΕΙΑΣ ΚΑΙ ΘΡΗΣΚΕΥΜΑΤΩΝ ΙΝΣΤΙΤΟΥΤΟ ΕΚΠΑΙΔΕΥΤΙΚΗΣ ΠΟΛΙΤΙΚΗΣ',
                     '## ΣΥΓΓΡΑΦΕΙΣ','## ΚΡΙΤΕΣ','## ΣΥΝΤΟΝΙΣΤΗΣ')

    paragraphs = paragraph_maker(text,maxpadding=1)
    paragraphs = paragraph_clean_image(paragraphs)
    paragraphs = paragraph_clean_dotlines(paragraphs)
    paragraphs = paragraph_remove_artifacts(paragraphs)
    paragraphs = paragraph_fix_broken_line(paragraphs)
    paragraphs = paragraph_merger(paragraphs,500,10)
    paragraphs = remove_diofantos(paragraphs)
    paragraphs = remove_funding(paragraphs)
    paragraphs = remove_legal_note_3966_2011(paragraphs)
    paragraphs = remove_taged_paragraphs(paragraphs,bibliography_tags)
    paragraphs = remove_taged_paragraphs(paragraphs,euritirio_tags)
    paragraphs = remove_taged_paragraphs(paragraphs,glossary_tags)
    paragraphs = remove_taged_paragraphs(paragraphs,church_tags)
    paragraphs = remove_taged_paragraphs(paragraphs,content_tags)
    paragraphs = remove_taged_paragraphs(paragraphs,printing_tags)
    paragraphs = all_paragraph_not_char_end(paragraphs,endings)
    paragraphs = remove_content_table_begin(paragraphs,print=True)
    paragraphs = paragraph_remove_artifacts(paragraphs)
    #paragraphs = remove_ending_chunk(paragraphs)
    
    t_file = 'test_'+file

    output_file_path = os.path.join('../test_output', t_file)
    try:
        with open(output_file_path, 'w', encoding='utf-8') as output_file:
            write_text(paragraphs,output_file)
    except Exception as e:
            print(f"Error writing to {output_file_path}: {e}")
            continue